<a href="https://colab.research.google.com/github/MiguelPartosa/Thesis-FOS-BinaryClass-WSD/blob/main/CBERT_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [28]:
!pip install transformers

In [29]:
# Importing stock ml libraries
import warnings
warnings.simplefilter('ignore')
import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn import metrics
import transformers
import torch
from torch.utils.data import Dataset, DataLoader, RandomSampler, SequentialSampler
from transformers import DistilBertTokenizer, DistilBertModel, BertTokenizer, BertModel
import logging
logging.basicConfig(level=logging.ERROR)
from torch import cuda

In [30]:
def hamming_score(y_true, y_pred, normalize=True, sample_weight=None):
    acc_list = []
    for i in range(y_true.shape[0]):
        set_true = set( np.where(y_true[i])[0] )
        set_pred = set( np.where(y_pred[i])[0] )
        tmp_a = None
        if len(set_true) == 0 and len(set_pred) == 0:
            tmp_a = 1
        else:
            tmp_a = len(set_true.intersection(set_pred))/\
                    float( len(set_true.union(set_pred)) )
        acc_list.append(tmp_a)
    return np.mean(acc_list)

In [31]:
#from google.colab import drive
#drive.mount('/content/drive')

In [32]:
#data = pd.read_excel('/content/drive/MyDrive/4th year/cbert/Transformed_Dataset.xlsx')
#data = pd.read_csv('/content/fosData - fosData(1).csv')

data = pd.read_excel('/content/Train_Set_WLabels.xlsx')

In [33]:
data

,Usage,Is FOS
0,Ang iyang kauban sa trabaho pirmi magpalapad o...,"[1,0]"
1,Nagpalit siya og balayronon sa papel sa tindah...,"[0,1]"
2,"""Ang silingan murag namiya og tai, kalit lang ...","[1,0]"
3,Ang bata nagdali sa pagdagan kay murag namiya ...,"[0,1]"
4,"Si Pedro makawat kaayo, dali ra siya madala bi...","[1,0]"
...,...,...
1283,Ang bata nalipay nga gidawat limpyo ang bag-on...,"[0,1]"
1284,Naa sila sa tunga sa ilang paglakaw padulong s...,"[1,0]"
1285,"""Pag-abot nako sa balay, akong gibutangan ug t...","[0,1]"
1286,Si Maria nagstruggle kay pirmi malimtan ang mg...,"[1,0]"


In [34]:
new_df = pd.DataFrame()
new_df['text'] = data['Usage']#post
new_df['labels'] = data['Is FOS'].values.tolist()

In [35]:
new_df.head()

,text,labels
0,Ang iyang kauban sa trabaho pirmi magpalapad o...,"[1,0]"
1,Nagpalit siya og balayronon sa papel sa tindah...,"[0,1]"
2,"""Ang silingan murag namiya og tai, kalit lang ...","[1,0]"
3,Ang bata nagdali sa pagdagan kay murag namiya ...,"[0,1]"
4,"Si Pedro makawat kaayo, dali ra siya madala bi...","[1,0]"


In [36]:
type(new_df['labels'][1]) #check data type

str

In [37]:
new_df['labels'][1]

'[0,1]'

In [38]:
#converts the labels column values from string to list and remove the characters , and [ and ] from the list

import ast

def convert_labels(df):
  """Converts the 'labels' column from string to list and removes special characters.
  """
  new_labels = []
  for labels in df['labels']:
    try:
      # Safely evaluate the string as a Python literal
      labels_list = ast.literal_eval(labels)
      # Remove unwanted characters
      cleaned_list = [str(item).replace('[', '').replace(']', '').replace(',', '').replace("'", "") for item in labels_list]
      new_labels.append(cleaned_list)
    except (SyntaxError, ValueError):
        # Handle cases where the string is not a valid list representation
        print(f"Invalid label string: {labels}")
        new_labels.append([])  # Or another appropriate default value
  return new_labels

# Example usage
new_df['labels'] = convert_labels(new_df)
print(new_df.head())

                                                text  labels
0  Ang iyang kauban sa trabaho pirmi magpalapad o...  [1, 0]
1  Nagpalit siya og balayronon sa papel sa tindah...  [0, 1]
2  "Ang silingan murag namiya og tai, kalit lang ...  [1, 0]
3  Ang bata nagdali sa pagdagan kay murag namiya ...  [0, 1]
4  Si Pedro makawat kaayo, dali ra siya madala bi...  [1, 0]


In [39]:
new_df.head()

,text,labels
0,Ang iyang kauban sa trabaho pirmi magpalapad o...,"[1, 0]"
1,Nagpalit siya og balayronon sa papel sa tindah...,"[0, 1]"
2,"""Ang silingan murag namiya og tai, kalit lang ...","[1, 0]"
3,Ang bata nagdali sa pagdagan kay murag namiya ...,"[0, 1]"
4,"Si Pedro makawat kaayo, dali ra siya madala bi...","[1, 0]"


In [40]:
type(new_df['labels'][0]) #check to see if values are in a list

list

In [41]:
for item in new_df['labels'][0]:
  print(type(item))

<class 'str'>
<class 'str'>


In [42]:
#change the datatype of the items in the list into int values

new_df['labels'] = new_df['labels'].apply(lambda x: [int(item) for item in x])


In [43]:
new_df['labels'][0]

[1, 0]

In [44]:
for item in new_df['labels'][0]:
  print(type(item))

<class 'int'>
<class 'int'>


In [45]:
from transformers import DistilBertTokenizer, DistilBertModel, BertTokenizer, BertModel

In [46]:
# Sections of config

# Defining some key variables that will be used later on in the training
MAX_LEN = 128
TRAIN_BATCH_SIZE = 32
VALID_BATCH_SIZE = 32
EPOCHS = 10
LEARNING_RATE = 2e-05
#use uncased distilbert tokenizer
#tokenizer = RobertaTokenizer.from_pretrained('dost-asti/BERT-ceb-cased', truncation=True, do_lower_case=False)
#tokenizer = BertTokenizer.from_pretrained('google-bert/bert-base-multilingual-cased', truncation=True, do_lower_case=False)
tokenizer = DistilBertTokenizer.from_pretrained('GianTan/CBERTo', truncation=True, do_lower_case=False)

In [47]:
class MultiLabelDataset(Dataset):

    def __init__(self, dataframe, tokenizer, max_len):
        self.tokenizer = tokenizer
        self.data = dataframe
        self.text = dataframe.text
        self.targets = self.data.labels
        self.max_len = max_len

    def __len__(self):
        return len(self.text)

    def __getitem__(self, index):
        text = str(self.text[index])
        text = " ".join(text.split())

        inputs = self.tokenizer.encode_plus(
            text,
            None,
            add_special_tokens=True,
            max_length=self.max_len,
            pad_to_max_length=True,
            return_token_type_ids=True
        )
        ids = inputs['input_ids']
        mask = inputs['attention_mask']
        token_type_ids = inputs["token_type_ids"]


        return {
            'ids': torch.tensor(ids, dtype=torch.long),
            'mask': torch.tensor(mask, dtype=torch.long),
            'token_type_ids': torch.tensor(token_type_ids, dtype=torch.long),
            'targets': torch.tensor(self.targets[index], dtype=torch.float)
        }

In [48]:
# Creating the dataset and dataloader for the neural network

train_size = 0.8
train_data=new_df.sample(frac=train_size,random_state=200)
test_data=new_df.drop(train_data.index).reset_index(drop=True)
train_data = train_data.reset_index(drop=True)


print("FULL Dataset: {}".format(new_df.shape))
print("TRAIN Dataset: {}".format(train_data.shape))
print("TEST Dataset: {}".format(test_data.shape))

training_set = MultiLabelDataset(train_data, tokenizer, MAX_LEN)
testing_set = MultiLabelDataset(test_data, tokenizer, MAX_LEN)

FULL Dataset: (1288, 2)
TRAIN Dataset: (1030, 2)
TEST Dataset: (258, 2)


In [49]:
train_data.head()

,text,labels
0,Kalas og bugo ang magdiskusyon ug walay resulta.,"[1, 0]"
1,"Si Juan bukhad og palad, bisan sa iyang kaliso...","[1, 0]"
2,"""Ayaw pag pangatulig luwag, klaroha imong mga ...","[1, 0]"
3,Ang libro nabasa ang tingog kay natagbaw sa tu...,"[0, 1]"
4,Ang ilang negosyo nga paabang og sakyanan kay ...,"[1, 0]"


In [50]:
training_set[0]

Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.


{'ids': tensor([   2,    1,  262, 1874,  126,  177, 1531, 2445,  840,  184,  427, 3304,
           20,    3,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0]),
 'mask': tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0,

In [51]:
train_params = {'batch_size': TRAIN_BATCH_SIZE,
                'shuffle': True,
                'num_workers': 0
                }

test_params = {'batch_size': VALID_BATCH_SIZE,
                'shuffle': True,
                'num_workers': 0
                }

training_loader = DataLoader(training_set, **train_params)
testing_loader = DataLoader(testing_set, **test_params)

In [27]:
device = 'cuda' if cuda.is_available() else 'cpu'
device

'cuda'

In [53]:
# Creating the customized model, by adding a drop out and a dense layer on top of distil bert to get the final output for the model.

class BertClass(torch.nn.Module):
    def __init__(self):
        super(BertClass, self).__init__()
        self.l1 = DistilBertModel.from_pretrained("GianTan/CBERTo")
        #self.l1 = RobertaModel.from_pretrained('dost-asti/BERT-ceb-cased')
        #self.l1 = BertModel.from_pretrained('google-bert/bert-base-multilingual-cased')

        self.pre_classifier = torch.nn.Linear(768, 768)
        self.dropout = torch.nn.Dropout(0.1)
        self.classifier = torch.nn.Linear(768, 2) # change this to 1

    def forward(self, input_ids, attention_mask, token_type_ids):
        output_1 = self.l1(input_ids=input_ids, attention_mask=attention_mask)
        hidden_state = output_1[0]
        pooler = hidden_state[:, 0]
        pooler = self.pre_classifier(pooler)
        pooler = torch.nn.Tanh()(pooler)
        pooler = self.dropout(pooler)
        output = self.classifier(pooler)
        return output

model = BertClass()
model.to(device)

BertClass(
  (l1): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30000, 768, padding_idx=0)
      (position_embeddings): Embedding(256, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)
            (lin1): Linear(in_feat

In [54]:
def loss_fn(outputs, targets):
    return torch.nn.BCEWithLogitsLoss()(outputs, targets)

In [55]:
optimizer = torch.optim.Adam(params =  model.parameters(), lr=LEARNING_RATE)

In [56]:
def train(epoch):
    model.train()
    for _,data in tqdm(enumerate(training_loader, 0)):
        ids = data['ids'].to(device, dtype = torch.long)
        mask = data['mask'].to(device, dtype = torch.long)
        token_type_ids = data['token_type_ids'].to(device, dtype = torch.long)
        targets = data['targets'].to(device, dtype = torch.float)

        outputs = model(ids, mask, token_type_ids)

        optimizer.zero_grad()
        loss = loss_fn(outputs, targets)
        if _%5000==0:
            print(f'Epoch: {epoch}, Loss:  {loss.item()}')

        loss.backward()
        optimizer.step()

In [57]:
for epoch in range(EPOCHS):
    train(epoch)

0it [00:00, ?it/s]

Epoch: 0, Loss:  0.7117286324501038


33it [00:10,  3.10it/s]
1it [00:00,  7.55it/s]

Epoch: 1, Loss:  0.46334031224250793


33it [00:09,  3.65it/s]
1it [00:00,  7.08it/s]

...

33it [00:09,  3.45it/s]
1it [00:00,  6.77it/s]

Epoch: 8, Loss:  0.008081598207354546


33it [00:09,  3.43it/s]
1it [00:00,  6.53it/s]

Epoch: 9, Loss:  0.015380497090518475


33it [00:09,  3.40it/s]


In [58]:
'''
def validation(testing_loader):
    model.eval()
    fin_targets=[]
    fin_outputs=[]
    i = 0
    with torch.no_grad():
        for _, data in tqdm(enumerate(testing_loader, 0)):
            ids = data['ids'].to(device, dtype = torch.long)
            mask = data['mask'].to(device, dtype = torch.long)
            token_type_ids = data['token_type_ids'].to(device, dtype = torch.long)
            targets = data['targets'].to(device, dtype = torch.float)
            outputs = model(ids, mask, token_type_ids)
            fin_targets.extend(targets.cpu().detach().numpy().tolist())
            fin_outputs.extend(torch.sigmoid(outputs).cpu().detach().numpy().tolist())
            print(f'{len(data)}:{data}:{fin_outputs[i]}')
            print("="*200)
            i = i + 1
    return fin_outputs, fin_targets
  '''

'\ndef validation(testing_loader):\n    model.eval()\n    fin_targets=[]\n    fin_outputs=[]\n    i = 0\n    with torch.no_grad():\n        for _, data in tqdm(enumerate(testing_loader, 0)):\n            ids = data[\'ids\'].to(device, dtype = torch.long)\n            mask = data[\'mask\'].to(device, dtype = torch.long)\n            token_type_ids = data[\'token_type_ids\'].to(device, dtype = torch.long)\n            targets = data[\'targets\'].to(device, dtype = torch.float)\n            outputs = model(ids, mask, token_type_ids)\n            fin_targets.extend(targets.cpu().detach().numpy().tolist())\n            fin_outputs.extend(torch.sigmoid(outputs).cpu().detach().numpy().tolist())\n            print(f\'{len(data)}:{data}:{fin_outputs[i]}\')\n            print("="*200)\n            i = i + 1\n    return fin_outputs, fin_targets\n  '

In [59]:
def validation(testing_loader):
    model.eval()
    fin_targets=[]
    fin_outputs=[]

    i = 0
    with torch.no_grad():
        for _, data in tqdm(enumerate(testing_loader, 0)):
            ids = data['ids'].to(device, dtype = torch.long)
            mask = data['mask'].to(device, dtype = torch.long)
            token_type_ids = data['token_type_ids'].to(device, dtype = torch.long)
            targets = data['targets'].to(device, dtype = torch.float)
            outputs = model(ids, mask, token_type_ids)
            fin_targets.extend(targets.cpu().detach().numpy().tolist())
            fin_outputs.extend(torch.sigmoid(outputs).cpu().detach().numpy().tolist())
            print(f'IDS:{ids}:{fin_outputs[i]}')
            print("="*200)
            i = i + 1
    return fin_outputs, fin_targets


In [60]:
testing_loader

In [61]:
testing_set[0]

{'ids': tensor([    2,     1,   234,  1447,   166,  1446, 13101, 11989,  2957,   262,
          3156,   166,   195,  2829,  4158,   630, 19169,   166,   557,    20,
             3,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,  

In [62]:
outputs, targets = validation(testing_loader)

final_outputs = np.array(outputs) >= 0.5

2it [00:00,  8.58it/s]

IDS:tensor([[   2,    1,  301,  ...,    0,    0,    0],
        [   2,    1,    1,  ...,    0,    0,    0],
        [   2,    1,    1,  ...,    0,    0,    0],
        ...,
        [   2,    1, 2322,  ...,    0,    0,    0],
        [   2,    1,  195,  ...,    0,    0,    0],
        [   2,    1,  234,  ...,    0,    0,    0]], device='cuda:0'):[0.0013508492847904563, 0.9988183379173279]
IDS:tensor([[   2,    1, 5976,  ...,    0,    0,    0],
        [   2,    1,  166,  ...,    0,    0,    0],
        [   2,    1,  195,  ...,    0,    0,    0],
        ...,
        [   2,    1,  195,  ...,    0,    0,    0],
        [   2,    1,  195,  ...,    0,    0,    0],
        [   2,    1,  199,  ...,    0,    0,    0]], device='cuda:0'):[0.9982808828353882, 0.0019010937539860606]


4it [00:00,  8.78it/s]

IDS:tensor([[    2,     1,     1,  ...,     0,     0,     0],
        [    2,     1,   195,  ...,     0,     0,     0],
        [    2,     1,     1,  ...,     0,     0,     0],
        ...,
        [    2,     1, 24320,  ...,     0,     0,     0],
        [    2,     8,     1,  ...,     0,     0,     0],
        [    2,     1,   674,  ...,     0,     0,     0]], device='cuda:0'):[0.002724874997511506, 0.9975675344467163]
IDS:tensor([[    2,     1,     1,  ...,     0,     0,     0],
        [    2,     1, 17576,  ...,     0,     0,     0],
        [    2,     1,    19,  ...,     0,     0,     0],
        ...,
        [    2,     8,     1,  ...,     0,     0,     0],
        [    2,     1,     1,  ...,     0,     0,     0],
        [    2,     1,   304,  ...,     0,     0,     0]], device='cuda:0'):[0.001590960891917348, 0.9986544847488403]


6it [00:00,  8.90it/s]

IDS:tensor([[   2,    1,  234,  ...,    0,    0,    0],
        [   2,    1,  192,  ...,    0,    0,    0],
        [   2,    1,  199,  ...,    0,    0,    0],
        ...,
        [   2,    1,    1,  ...,    0,    0,    0],
        [   2,    1,  702,  ...,    0,    0,    0],
        [   2,    1, 1476,  ...,    0,    0,    0]], device='cuda:0'):[0.002507996978238225, 0.9977803826332092]
IDS:tensor([[    2,     8,     1,  ...,     0,     0,     0],
        [    2,     8,     1,  ...,     0,     0,     0],
        [    2,     1,   326,  ...,     0,     0,     0],
        ...,
        [    2,     1,  1476,  ...,     0,     0,     0],
        [    2,     1, 10694,  ...,     0,     0,     0],
        [    2,     1,   234,  ...,     0,     0,     0]], device='cuda:0'):[0.0017682784236967564, 0.9982385635375977]


9it [00:00,  9.62it/s]

IDS:tensor([[  2,   1,   1,  ...,   0,   0,   0],
        [  2,   1,   1,  ...,   0,   0,   0],
        [  2,   1, 269,  ...,   0,   0,   0],
        ...,
        [  2,   1,   1,  ...,   0,   0,   0],
        [  2,   1, 234,  ...,   0,   0,   0],
        [  2,   1, 195,  ...,   0,   0,   0]], device='cuda:0'):[0.9720527529716492, 0.024868303909897804]
IDS:tensor([[  2,   1, 618,  ...,   0,   0,   0],
        [  2,   1, 199,  ...,   0,   0,   0],
        [  2,   1, 195,  ...,   0,   0,   0],
        ...,
        [  2,   1, 234,  ...,   0,   0,   0],
        [  2,   1,   1,  ...,   0,   0,   0],
        [  2,   1, 195,  ...,   0,   0,   0]], device='cuda:0'):[0.0025370267685502768, 0.9977310299873352]
IDS:tensor([[    2,     1,   319,   177,   892,  1244,   593,   166, 11385,   303,
          3434,   177,   234, 23526,  3207,    20,     3,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,

In [63]:
len(final_outputs)

258

In [64]:
final_outputs[2]

array([False,  True])

In [65]:
val_hamming_loss = metrics.hamming_loss(targets, final_outputs)
val_hamming_score = hamming_score(np.array(targets), np.array(final_outputs))

print(f"Hamming Score = {val_hamming_score}")
print(f"Hamming Loss = {val_hamming_loss}")

Hamming Score = 0.8875968992248062
Hamming Loss = 0.1124031007751938


In [66]:
from sklearn.metrics import f1_score

In [67]:
#f1 score test
score_dat = f1_score(np.array(targets),np.array(final_outputs),average='weighted')
print(score_dat)

0.8875377743728077


In [68]:
from sklearn.metrics import accuracy_score

In [69]:
#accuracy score test
y_pred = np.array(targets)
y_true = np.array(final_outputs)

accuracy_score(y_true, y_pred)

0.8875968992248062

In [70]:
test_set = MultiLabelDataset(test_data, tokenizer, MAX_LEN)
testing_params = {'batch_size': TRAIN_BATCH_SIZE,
               'shuffle': False,
               'num_workers': 2
                }
test_loader = DataLoader(test_set, **testing_params)

In [71]:
all_test_pred = []

def test(epoch):
    model.eval()

    with torch.inference_mode():

        for _, data in tqdm(enumerate(test_loader, 0)):


            ids = data['ids'].to(device, dtype=torch.long)
            mask = data['mask'].to(device, dtype=torch.long)
            token_type_ids = data['token_type_ids'].to(device, dtype=torch.long)
            outputs = model(ids, mask, token_type_ids)
            probas = torch.sigmoid(outputs)

            all_test_pred.append(probas)
    return probas

In [72]:
probas = test(model)

9it [00:00, 10.02it/s]


In [73]:
all_test_pred = torch.cat(all_test_pred)

In [74]:
all_test_pred

tensor([[0.9768, 0.0284],
        [0.9975, 0.0027],
        [0.0014, 0.9988],
        [0.0014, 0.9988],
        [0.9986, 0.0014],
        [0.0025, 0.9975],
        [0.6035, 0.4442],
        [0.9986, 0.0014],
        [0.0015, 0.9986],
        [0.0021, 0.9981],
        [0.0015, 0.9986],
        [0.9986, 0.0014],
        [0.5151, 0.4816],
        [0.9987, 0.0013],
        [0.0014, 0.9988],
        [0.0034, 0.9974],
        [0.9979, 0.0021],
        [0.0115, 0.9881],
        [0.0015, 0.9987],
        [0.9988, 0.0013],
        [0.0016, 0.9986],
        [0.9988, 0.0013],
        [0.9988, 0.0014],
        [0.9725, 0.0262],
        [0.9987, 0.0013],
        [0.9983, 0.0015],
        [0.0015, 0.9988],
        [0.9985, 0.0014],
        [0.0014, 0.9987],
        [0.0507, 0.9386],
        [0.0017, 0.9986],
        [0.9984, 0.0016],
        [0.0015, 0.9986],
        [0.0014, 0.9988],
        [0.9987, 0.0013],
        [0.0038, 0.9965],
        [0.9976, 0.0020],
        [0.0013, 0.9988],
        [0.0

In [75]:
submit_df = test_data.copy()

In [76]:
submit_df

,text,labels
0,Ang iyang kauban sa trabaho pirmi magpalapad o...,"[1, 0]"
1,"Ang iyang amigo gwapo kon magtalikod, pero bat...","[1, 0]"
2,"Siya miingon nga ang masuya madeads, og nagtuo...","[0, 1]"
3,Ang bantay pari naglingkod sa daplin sa altar ...,"[0, 1]"
4,Si Maria lunod patay sa pagpanalipod sa iyang ...,"[1, 0]"
...,...,...
253,"""Nadakpan na ba ang nagtupi nimo kay wala man ...","[0, 1]"
254,Ang mga awas og palad kasagaran maglisod sa pa...,"[1, 0]"
255,Ang mga estudyante na gi-indian sa ilang maest...,"[1, 0]"
256,"""Naghinaw ug maayo si Juan human mo-kaon.""","[0, 1]"


In [77]:
label_columns = ["non-literal", "literal"]

In [78]:
for i,name in enumerate(label_columns):

    submit_df[name] = all_test_pred[:, i].cpu()
    submit_df.head()

In [79]:
submit_df

,text,labels,non-literal,literal
0,Ang iyang kauban sa trabaho pirmi magpalapad o...,"[1, 0]",0.976768,0.028425
1,"Ang iyang amigo gwapo kon magtalikod, pero bat...","[1, 0]",0.997549,0.002702
2,"Siya miingon nga ang masuya madeads, og nagtuo...","[0, 1]",0.001386,0.998808
3,Ang bantay pari naglingkod sa daplin sa altar ...,"[0, 1]",0.001424,0.998827
4,Si Maria lunod patay sa pagpanalipod sa iyang ...,"[1, 0]",0.998638,0.001447
...,...,...,...,...
253,"""Nadakpan na ba ang nagtupi nimo kay wala man ...","[0, 1]",0.005399,0.994015
254,Ang mga awas og palad kasagaran maglisod sa pa...,"[1, 0]",0.998690,0.001341
255,Ang mga estudyante na gi-indian sa ilang maest...,"[1, 0]",0.990867,0.009843
256,"""Naghinaw ug maayo si Juan human mo-kaon.""","[0, 1]",0.039562,0.961833


In [80]:
submit_df.to_csv('predictions.csv')

In [81]:
#using the model

# Preprocess the input text
'''

input_text = "lantaw sa ko ug tv lol"
encoded_text = tokenizer.encode_plus(
    input_text,
    None,
    add_special_tokens=True,
    max_length=MAX_LEN,
    pad_to_max_length=True,
    return_token_type_ids=True
)

# Convert the input to tensors
input_ids = torch.tensor(encoded_text['input_ids']).unsqueeze(0)
input_mask = torch.tensor(encoded_text['attention_mask']).unsqueeze(0)
segment_ids = torch.tensor(encoded_text['token_type_ids']).unsqueeze(0)

# Move tensors to the device
input_ids = input_ids.to(device)
input_mask = input_mask.to(device)
segment_ids = segment_ids.to(device)

# Make predictions
with torch.no_grad():
    outputs = model(input_ids, input_mask, segment_ids)

# Apply sigmoid activation function
outputs = torch.sigmoid(outputs)

# Convert the outputs to numpy array
outputs = outputs.cpu().detach().numpy()

# Print the predictions
print(outputs)

'''


'\n\ninput_text = "lantaw sa ko ug tv lol"\nencoded_text = tokenizer.encode_plus(\n    input_text,\n    None,\n    add_special_tokens=True,\n    max_length=MAX_LEN,\n    pad_to_max_length=True,\n    return_token_type_ids=True\n)\n\n# Convert the input to tensors\ninput_ids = torch.tensor(encoded_text[\'input_ids\']).unsqueeze(0)\ninput_mask = torch.tensor(encoded_text[\'attention_mask\']).unsqueeze(0)\nsegment_ids = torch.tensor(encoded_text[\'token_type_ids\']).unsqueeze(0)\n\n# Move tensors to the device\ninput_ids = input_ids.to(device)\ninput_mask = input_mask.to(device)\nsegment_ids = segment_ids.to(device)\n\n# Make predictions\nwith torch.no_grad():\n    outputs = model(input_ids, input_mask, segment_ids)\n\n# Apply sigmoid activation function\noutputs = torch.sigmoid(outputs)\n\n# Convert the outputs to numpy array\noutputs = outputs.cpu().detach().numpy()\n\n# Print the predictions\nprint(outputs)\n\n'

Individual test

In [95]:
#using the model, non batches

# Preprocess the input text
input_text = "Ming-aw ang bulan nga nagtan-aw kanako sa gabii."
encoded_text = tokenizer.encode_plus(
    input_text,
    None,
    add_special_tokens=True,
    max_length=MAX_LEN,
    pad_to_max_length=True,
    return_token_type_ids=True
)

# Convert the input to tensors
input_ids = torch.tensor(encoded_text['input_ids']).unsqueeze(0)
input_mask = torch.tensor(encoded_text['attention_mask']).unsqueeze(0)
segment_ids = torch.tensor(encoded_text['token_type_ids']).unsqueeze(0)

# Move tensors to the device
input_ids = input_ids.to(device)
input_mask = input_mask.to(device)
segment_ids = segment_ids.to(device)

# Make predictions
with torch.no_grad():
    outputs = model(input_ids, input_mask, segment_ids)

# Apply sigmoid activation function
outputs = torch.sigmoid(outputs)
print(outputs.cpu())
# Convert the outputs to numpy array
outputs = outputs.cpu().detach().numpy()

# Print the predictions
print(outputs)

tensor([[0.0052, 0.9949]])
[[0.00516707 0.99494934]]
